# Trabalho Prático: Aprendizado de Máquina (Supervisionado e Não Supervisionado)
**Dataset:** QSAR Biodegradation (1054 amostras, 41 features moleculares)
**Objetivo:** Classificação binária de substâncias químicas (Prontamente Biodegradável - RB vs Não Prontamente Biodegradável - NRB).

## **Pipeline de Pré-processamento Avançado:**
1. Divisão Estratificada Imediata (70/15/15) para evitar *Data Leakage*.
2. Seleção de Features (Top 20) com Random Forest.
3. Tratamento Multivariado de Outliers com Isolation Forest (apenas no Treino).
4. Normalização Estatística (Z-Score).
5. Balanceamento de Classes com SMOTE (apenas no Treino).

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Divisão
from sklearn.model_selection import train_test_split, PredefinedSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from imblearn.over_sampling import SMOTE

# Modelos Supervisionados
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# Métricas de Avaliação
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Configurações visuais
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
import warnings
warnings.filterwarnings('ignore')

## 1. Carregamento, Limpeza Básica e Divisão Imediata
Para garantir a integridade do protocolo experimental, dividimos os dados em Treino (70%), Validação (15%) e Teste (15%) antes de aplicar qualquer técnica de redução ou tratamento de outliers.

In [ ]:
# Carregamento e remoção de duplicatas/nulos
df_raw = pd.read_csv('biodeg_normalizado.csv').drop_duplicates().dropna()

# Separação de Features e Target
target_col = 'experimental class'
X_raw = df_raw.drop(columns=[target_col])
y_raw = df_raw[target_col]

# Codificação da Variável Alvo
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
print(f"Mapeamento do Target: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

# Divisão: 30% para Temp (que será Validação e Teste) e 70% para Treino
X_train, X_temp, y_train, y_temp = train_test_split(
    X_raw, y_encoded, test_size=0.30, stratify=y_encoded, random_state=42
)

# Divisão: 50% do Temp para Validação (15% do total) e 50% para Teste (15% do total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Treino: {X_train.shape[0]} amostras | Validação: {X_val.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")

## 2. Seleção de Features (Redução de Dimensionalidade)
Utilizamos uma Random Forest apenas nos dados de treino para identificar as 20 características moleculares mais importantes, aplicando o Princípio da Parcimônia e reduzindo o ruído do dataset.

In [ ]:
print("--- Extraindo as 20 Features mais importantes ---")
# Treinamos a Random Forest (com class_weight para não ignorar a classe menor)
rf_selector = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_selector.fit(X_train, y_train)

# Coletando a importância
importancias = pd.DataFrame({'Feature': X_train.columns, 'Imp': rf_selector.feature_importances_})
importancias = importancias.sort_values(by='Imp', ascending=False)
top_20 = importancias.head(20)['Feature'].tolist()

# Gráfico para o Relatório
plt.figure(figsize=(10, 8))
sns.barplot(x='Imp', y='Feature', data=importancias.head(20), palette='viridis')
plt.title('Top 20 Features Mais Importantes para Biodegradabilidade')
plt.xlabel('Grau de Importância')
plt.ylabel('Descritor Molecular')
plt.tight_layout()
plt.show()

# Reduzimos todas as partições para as 20 colunas selecionadas
X_train_20 = X_train[top_20]
X_val_20 = X_val[top_20]
X_test_20 = X_test[top_20]
print("Dimensões reduzidas para 20 features com sucesso!")

## 3. Remoção de Outliers Multivariada (Isolation Forest)
Removeremos anomalias globais (5% mais extremos) **apenas no conjunto de treinamento**, preservando o conjunto de Teste como uma representação fiel das incertezas do mundo real.

In [ ]:
print("--- Removendo Outliers (Isolation Forest) do Treino ---")
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
outlier_labels = iso_forest.fit_predict(X_train_20)

mask_inliers = (outlier_labels == 1)
X_train_clean = X_train_20[mask_inliers]
y_train_clean = y_train[mask_inliers]

print(f"Amostras removidas (anomalias): {len(y_train) - len(y_train_clean)}")
print(f"Novo tamanho do conjunto de treino: {X_train_clean.shape[0]} amostras")

## 4. Normalização (Z-Score) e Balanceamento (SMOTE)
Com os dados de treino limpos, calculamos a média e o desvio padrão para normalizar todos os conjuntos. Em seguida, aplicamos o SMOTE no Treino para balancear as classes (50/50).

In [ ]:
print("--- Normalizando os dados (Z-Score) ---")
scaler = StandardScaler()

# O fit (aprendizado) ocorre APENAS no Treino limpo
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_clean), columns=top_20)

# O transform é aplicado passivamente na Validação e Teste
X_val_scaled = pd.DataFrame(scaler.transform(X_val_20), columns=top_20)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_20), columns=top_20)

print("--- Aplicando SMOTE no Treino ---")
smote = SMOTE(random_state=42)
X_train_final, y_train_final = smote.fit_resample(X_train_scaled, y_train_clean)
print(f"Novo tamanho do Treino após SMOTE: {X_train_final.shape[0]} amostras (Balanceado)")

## 5. Configuração da Avaliação e Modelagem Supervisionada
Usaremos o `PredefinedSplit` para forçar o `GridSearchCV` a validar as métricas de hiperparâmetros estritamente nos nossos 15% de Validação estáticos.

In [ ]:
# Preparação do PredefinedSplit
X_train_val = pd.concat([X_train_final, X_val_scaled], axis=0)
y_train_val = np.concatenate([y_train_final, y_val])

test_fold = np.zeros(X_train_val.shape[0])
test_fold[:X_train_final.shape[0]] = -1
ps = PredefinedSplit(test_fold)

# Dicionário de resultados e Função de Avaliação
resultados_modelos = {}

def avaliar_modelo(nome, modelo, params):
    print(f"\n{'='*60}\n--- Treinando e Otimizando: {nome} ---\n{'='*60}")
    grid = GridSearchCV(modelo, params, cv=ps, scoring='f1_macro', n_jobs=-1)
    grid.fit(X_train_val, y_train_val)

    best_model = grid.best_estimator_
    print(f"Melhores Hiperparâmetros: {grid.best_params_}")

    # Avaliação Final no Conjunto de Teste Intocado
    y_pred = best_model.predict(X_test_scaled)

    # Exibição dos Relatórios
    print("\nRelatório de Classificação no Conjunto de TESTE:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

    plt.figure(figsize=(5,4))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'Matriz de Confusão - {nome}')
    plt.ylabel('Classe Real')
    plt.xlabel('Classe Predita')
    plt.show()
    return best_model

## 6. Treinamento dos Modelos
Execução da Árvore de Decisão, K-Nearest Neighbors (KNN) e Rede Neural Artificial (MLP) com otimização dos melhores hiperparâmetros baseados na redução de features e tratamento de anomalias.

In [ ]:
# 1. Decision Tree
param_dt = {'criterion': ['gini', 'entropy'], 'max_depth': [5, 10, 15, 20, None],
            'min_samples_split': [2, 5, 10, 20], 'min_samples_leaf': [1, 5, 10], 'class_weight': [None, 'balanced']}
dt_model = avaliar_modelo("Decision Tree", DecisionTreeClassifier(random_state=42), param_dt)

# 2. KNN
param_knn = {'n_neighbors': [3, 5, 7, 9, 11, 15, 19], 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan']}
knn_model = avaliar_modelo("KNN", KNeighborsClassifier(), param_knn)

# 3. Rede Neural (MLP)
param_mlp = {'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
             'activation': ['relu', 'tanh', 'logistic'], 'alpha': [0.0001, 0.001, 0.01, 0.1], 'learning_rate_init': [0.001, 0.01]}
mlp_model = avaliar_modelo("Rede Neural (MLP)", MLPClassifier(max_iter=1000, random_state=42, early_stopping=False), param_mlp)

## 7. Modelagem Não Supervisionada (K-Means)
Vamos esquecer temporariamente os rótulos originais (`y`) e verificar como os dados moleculares se agrupam naturalmente. Usaremos toda a base padronizada (`X_scaled`).

Conforme especificado, iteraremos com $K$ variando de 2 a 10 e utilizaremos duas abordagens em conjunto para decidir o melhor $K$:
1. **Método do Cotovelo (Inércia / WCSS):** Busca o ponto onde a adição de novos clusters deixa de reduzir significativamente a variância interna.
2. **Silhouette Score:** Mede o quão similar um objeto é ao seu próprio cluster em comparação com outros clusters (varia de -1 a 1, sendo quanto mais próximo de 1, melhor).

In [ ]:
inercia = []
silhouette_scores = []
K_range = range(2, 11) # K entre 2 e 10

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)

    inercia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plotando os Gráficos Lado a Lado
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Gráfico 1: Cotovelo (Inércia)
ax[0].plot(K_range, inercia, marker='o', color='blue', linestyle='--')
ax[0].set_title('Método do Cotovelo (Inércia)')
ax[0].set_xlabel('Número de Clusters (k)')
ax[0].set_ylabel('Inércia (WCSS)')
ax[0].set_xticks(K_range)

# Gráfico 2: Silhouette Score
ax[1].plot(K_range, silhouette_scores, marker='s', color='green', linestyle='-')
ax[1].set_title('Silhouette Score')
ax[1].set_xlabel('Número de Clusters (k)')
ax[1].set_ylabel('Score de Silhueta')
ax[1].set_xticks(K_range)

plt.tight_layout()
plt.show()

### Análise e Execução do Melhor K
Em problemas binários, frequentemente testamos `K=2` para ver se os clusters refletem as classes originais (RB e NRB). No entanto, deves inspecionar visualmente o Silhouette e o Cotovelo. Para fins didáticos e baseando-se no comportamento comum deste dataset, aplicaremos `K=2` e compararemos os clusters formados de forma não-supervisionada com os rótulos reais.

In [ ]:
# Supondo k ótimo (exemplo: k=2 baseado na semântica do problema ou nos gráficos)
melhor_k = 2
print(f"Treinando K-Means com k={melhor_k}...")

kmeans_final = KMeans(n_clusters=melhor_k, random_state=42, n_init=10)
clusters_pred = kmeans_final.fit_predict(X_scaled)

# Comparando os clusters descobertos com a classe experimental real
df_comparacao = pd.DataFrame({
    'Classe_Real': label_encoder.inverse_transform(y_encoded),
    'Cluster_Atribuido': clusters_pred
})

crosstab = pd.crosstab(df_comparacao['Classe_Real'], df_comparacao['Cluster_Atribuido'],
                       rownames=['Classe Original'], colnames=['Cluster Encontrado'])
print("\nTabela de Contingência (Original vs Cluster):")
display(crosstab)

print("\nNota: Como o aprendizado é não-supervisionado, os rótulos do cluster (0 e 1) "
      "podem estar invertidos em relação aos rótulos originais. O que importa é a pureza "
      "de cada cluster (se ele isolou bem a maioria das instâncias RB de um lado e NRB do outro).")

## 8. Visualização dos Clusters (Redução de Dimensionalidade com PCA)
O dataset possui 41 dimensões, o que impede a visualização direta. Usaremos a Análise de Componentes Principais (PCA) para comprimir as 41 features em 2 componentes e enxergar a distribuição espacial dos clusters formados pelo K-Means.

In [ ]:
# Aplicando PCA para reduzir para 2 dimensões
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Criando dataframe para plotagem
df_pca = pd.DataFrame(data = X_pca, columns = ['Componente 1', 'Componente 2'])
df_pca['Cluster'] = clusters_pred
df_pca['Classe Real'] = label_encoder.inverse_transform(y_encoded)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Clusters do K-Means
sns.scatterplot(x='Componente 1', y='Componente 2', hue='Cluster',
                palette='Set1', data=df_pca, ax=ax[0], alpha=0.7)
ax[0].set_title('Clusters encontrados pelo K-Means')

# Plot 2: Classes Reais
sns.scatterplot(x='Componente 1', y='Componente 2', hue='Classe Real',
                palette='Set2', data=df_pca, ax=ax[1], alpha=0.7)
ax[1].set_title('Distribuição Real das Classes (RB vs NRB)')

plt.tight_layout()
plt.show()

print(f"Variância explicada pelos 2 componentes principais: {pca.explained_variance_ratio_.sum()*100:.2f}%")